# Pipeline Sinh Dataset Can Thiep Van Toc (Kubric GPU) -> Hugging Face

Notebook tu dong hoa toan bo quy trinh tren **MoLab** (https://molab.marimo.io/) hoac Google Colab / Jupyter:

1. Cai dat va import day du cac package can thiet.
2. Clone repository `p1neapplechoco/kubric` (kem co che tu dong khoi phuc file neu thieu).
3. Cai dat Docker qua `apt` va cau hinh NVIDIA Container Toolkit kem NVIDIA CDI cho GPU.
4. Cai dat Python 3.11 va Blender 4.2 LTS (`bpy==4.2.0` voi kernel Cycles OptiX/CUDA) trong Docker va Native.
5. Sinh dataset voi cau hinh can thiep:
   - Khoi luong: Co dinh (1.0 kg cho moi vat the dong).
   - Van toc: 1 huong duy nhat (chi cap van toc dau o buoc t=0 cho subject, khong co ngoai luc sau do).
   - Vat lieu: Sampling tu 6 ho vat lieu (metal, rubber, plastic, ceramic, wood, stone) ghep cap voi he so ma sat va dan hoi.
   - So luong: 4-6 vat the (1 subject, 1-3 vat the tuong tac trong corridor va cac bystander ngoai corridor).
   - Chuyen dong: Vua truot vua lan (sliding and rolling).
   - Camera: Co dinh goc quay trong suot video, da dang giua cac video khac nhau.
   - 3 nhanh mo phong: `factual`, `counterfactual` (ap van toc dau moi), `subject_removed` (bo subject).
6. Xuat du 4 loai output: Video (RGB), Mask (segmentation), Graph (do thi tiep xuc), Tracking (toa do 2D/3D).
7. Upload dataset len Hugging Face Hub bang Access Token.

## 1. Cai dat Package & Thiet lap Import

In [ ]:
# Cai dat cac package ho tro dieu phoi
!pip install --quiet "huggingface_hub>=0.24" "pyyaml>=6" "pandas>=2" "marimo>=0.13"

import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import yaml
from huggingface_hub import HfApi, HfFolder

print("Python version (Notebook):", sys.version.split()[0])
print("Imported successfully.")

## 2. Cau hinh Tham so

In [ ]:
# Thong tin repository
REPO_URL = "https://github.com/p1neapplechoco/kubric"
REPO_REF = "artifacts/data-generation-notebook"  # Nhanh chua pipeline can thiep van toc
WORKDIR = "/tmp/kubric-work"

# Tham so sinh dataset
MASTER_SEED = 0
SCENE_COUNT = 4              # So luong scene can sinh (moi scene gom 3 nhanh)
RESOLUTION = 256             # Do phan giai video (128, 256, 384, 512)
SAMPLES_PER_PIXEL = 64       # Cycles spp
REQUIRE_GPU = True           # Bat buoc dung GPU cho Cycles (bao loi neu khong co GPU)
PREFER_DOCKER = True         # Uu tien dung Docker neu moi truong cho phep

# Thong tin upload len Hugging Face
HF_REPO_ID = ""              # Dinh dang: username/dataset-name (vi du: john/kubric-dataset)
HF_TOKEN = ""                # Access token co quyen WRITE tren Hugging Face
HF_PRIVATE = True            # Dat dataset o che do rieng tu

## 3. Clone Repository & Tu dong Dam bao File Dockerfile / Cau hinh

In [ ]:
work_path = Path(WORKDIR)
work_path.mkdir(parents=True, exist_ok=True)
repo_dir = work_path / "kubric"

if not (repo_dir / ".git").exists():
    print(f"Dang clone {REPO_URL} (branch: {REPO_REF})...")
    res = subprocess.run(f"git clone --depth 1 --branch {REPO_REF} {REPO_URL} {repo_dir}", shell=True)
    if res.returncode != 0:
        print("Clone branch that bai, tien hanh clone toan bo repo va checkout...")
        subprocess.run(f"git clone {REPO_URL} {repo_dir}", shell=True, check=True)
        subprocess.run(f"cd {repo_dir} && git checkout {REPO_REF}", shell=True, check=False)
else:
    print("Repository da ton tai. Dang cap nhat...")
    subprocess.run(f"cd {repo_dir} && git fetch --all && git checkout {REPO_REF} && git pull --ff-only || true", shell=True)

# Tu dong tao file Dockerfile phong truong hop repo remote chua push
dockerfile_path = repo_dir / "docker" / "KubricGPU.Dockerfile"
if not dockerfile_path.exists():
    dockerfile_path.parent.mkdir(parents=True, exist_ok=True)
    dockerfile_content = """FROM nvidia/cuda:12.4.1-runtime-ubuntu22.04
ENV DEBIAN_FRONTEND=noninteractive PYTHONUNBUFFERED=1 TF_CPP_MIN_LOG_LEVEL=3 KUBRIC_USE_GPU=true NVIDIA_DRIVER_CAPABILITIES=compute,utility,graphics
RUN apt-get update && apt-get install -y --no-install-recommends software-properties-common ca-certificates curl gnupg git ffmpeg \
    && add-apt-repository -y ppa:deadsnakes/ppa \
    && apt-get update && apt-get install -y --no-install-recommends \
      python3.11 python3.11-dev python3.11-venv python3.11-distutils \
      libx11-6 libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libglu1-mesa \
      libsm6 libice6 libxkbcommon0 libegl1 libgomp1 libopenexr-dev \
    && rm -rf /var/lib/apt/lists/*
RUN curl -sS https://bootstrap.pypa.io/get-pip.py | python3.11 \
    && ln -sf /usr/bin/python3.11 /usr/local/bin/python \
    && ln -sf /usr/bin/python3.11 /usr/local/bin/python3 \
    && python -m pip install --no-cache-dir --upgrade pip wheel setuptools
WORKDIR /kubric
COPY requirements_render.txt .
RUN python -m pip install --no-cache-dir -r requirements_render.txt
ENV PYTHONPATH=/kubric
CMD ["python", "-c", "import bpy; print('bpy ready')"]
"""
    dockerfile_path.write_text(dockerfile_content)
    print("Da tu dong tao file docker/KubricGPU.Dockerfile san sang.")

print("Repository va cac file can thiet da san sang tai:", repo_dir)

## 4. Cai dat Docker & NVIDIA Container Toolkit (qua apt)

In [ ]:
install_docker_cmd = """
set -e
PRE=""
if [ "$(id -u)" -ne 0 ]; then
    if command -v sudo >/dev/null 2>&1; then PRE="sudo "; fi
fi

if command -v apt-get >/dev/null 2>&1 && [ -n "$PRE" -o "$(id -u)" -eq 0 ]; then
    echo "[1/3] Cai dat docker.io..."
    $PRE apt-get update -y
    $PRE DEBIAN_FRONTEND=noninteractive apt-get install -y docker.io curl gnupg ca-certificates

    echo "[2/3] Cau hinh NVIDIA Container Toolkit repo..."
    curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | $PRE gpg --yes --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
    curl -sL https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | \
      sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
      $PRE tee /etc/apt/sources.list.d/nvidia-container-toolkit.list >/dev/null

    echo "[3/3] Cai dat nvidia-container-toolkit va tao CDI spec..."
    $PRE apt-get update -y
    $PRE DEBIAN_FRONTEND=noninteractive apt-get install -y nvidia-container-toolkit
    $PRE nvidia-ctk runtime configure --runtime=docker
    $PRE mkdir -p /etc/cdi
    $PRE nvidia-ctk cdi generate --output=/etc/cdi/nvidia.yaml

    # Khoi dong docker daemon neu chua chay
    if ! docker info >/dev/null 2>&1; then
        $PRE systemctl restart docker 2>/dev/null || nohup $PRE dockerd > /tmp/dockerd.log 2>&1 &
        sleep 5
    fi
    docker info | grep -E "Server Version|Runtimes" || true
else
    echo "Moi truong khong ho tro apt / khong co quyen sudo. Se dung trinh chay native o buoc 5."
fi
"""
subprocess.run(install_docker_cmd, shell=True, check=False)

## 5. Setup Python 3.11 va Blender 4.2 (GPU)

Buoc nay tu dong thiet lap trinh thuc thi:
- **Che do Docker GPU**: Build `docker/KubricGPU.Dockerfile` (chua san Python 3.11 + Blender 4.2 LTS `bpy` ho tro CUDA/OptiX).
- **Che do Native Fallback**: Neu Docker khong khoi dong duoc (vi du sandbox MoLab), tu dong tai ban standalone Python 3.11 bang `uv` va cai dat dung phien ban `bpy==4.2.0` cung dependencies trong `requirements_render.txt`.

In [ ]:
IMAGE_TAG = "kubric-gpu:4.2"
runner = None
runner_kind = None

def check_docker_gpu():
    if not PREFER_DOCKER or not shutil.which("docker"):
        return False
    code = subprocess.run("docker run --rm --gpus all nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi -L",
                          shell=True, capture_output=True, text=True)
    return code.returncode == 0 and "GPU" in code.stdout

if check_docker_gpu():
    print("Phat hien Docker GPU hop le. Dang build KubricGPU Docker image...")
    subprocess.run(f"docker build -f docker/KubricGPU.Dockerfile -t {IMAGE_TAG} .",
                   shell=True, cwd=str(repo_dir), check=True)
    uid = os.getuid() if hasattr(os, "getuid") else 0
    gid = os.getgid() if hasattr(os, "getgid") else 0
    runner = (
        f"docker run --rm --gpus all --user {uid}:{gid} -e HOME=/tmp -e PYTHONPATH={repo_dir} "
        f"-e TF_CPP_MIN_LOG_LEVEL=3 --volume {work_path}:{work_path} --workdir {repo_dir} {IMAGE_TAG} python"
    )
    runner_kind = "docker"
else:
    print("Chuyen sang trinh chay Python 3.11 Native (dung uv de tai Python 3.11 va cai Blender bpy)...")
    if not shutil.which("uv"):
        subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)
        os.environ["PATH"] = str(Path.home() / ".local" / "bin") + os.pathsep + os.environ["PATH"]
    
    # Cai thu vien do hoa can thiet cho bpy tren Ubuntu
    subprocess.run("sudo apt-get update -y && sudo apt-get install -y --no-install-recommends "
                   "libx11-6 libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libglu1-mesa "
                   "libsm6 libice6 libxkbcommon0 libegl1 libgomp1 ffmpeg", shell=True, check=False)
    
    venv_dir = repo_dir / ".venv-render"
    if not (venv_dir / "bin" / "python").exists():
        subprocess.run(f"uv venv {venv_dir} --python 3.11", shell=True, check=True)
    
    print("Dang cai dat bpy==4.2.0 va cac dependencies pinned vao Python 3.11...")
    subprocess.run(f"uv pip install --python {venv_dir / 'bin' / 'python'} -r requirements_render.txt",
                   shell=True, cwd=str(repo_dir), check=True)
    runner = str(venv_dir / "bin" / "python")
    runner_kind = "native"

print(f"Trinh chay da san sang ({runner_kind}): {runner}")

# Kiem tra Cycles GPU ben trong Blender
probe_script = (
    "import bpy, json; "
    "p = bpy.context.preferences.addons['cycles'].preferences; "
    "devices = {b: [d.name for d in p.get_devices_for_type(b)] for b in ('OPTIX', 'CUDA')}; "
    "print('BLENDER_PROBE ' + json.dumps({'bpy_version': bpy.app.version_string, 'devices': devices}))"
)
probe_res = subprocess.run(f'{runner} -c "{probe_script}"', shell=True, cwd=str(repo_dir), capture_output=True, text=True)
for line in probe_res.stdout.splitlines():
    if line.startswith("BLENDER_PROBE "):
        print("Ket qua kiem tra Blender Cycles:", line[len("BLENDER_PROBE "):])

## 6. Chay Kiem thu Vat ly (Smoke Test, 1 Scene, Khong Render)

In [ ]:
smoke_out = work_path / "smoke_test"
smoke_cmd = f"{runner} scripts/build_velocity_dataset.py --output {smoke_out} --seed 12345 --count 1 --no-render"
env_vars = dict(os.environ, PYTHONPATH=str(repo_dir), TF_CPP_MIN_LOG_LEVEL="3")
subprocess.run(smoke_cmd, shell=True, cwd=str(repo_dir), env=env_vars, check=True)

qc_path = smoke_out / "instances" / "000000" / "qc.json"
gt_path = smoke_out / "instances" / "000000" / "ground_truth.json"
qc = json.loads(qc_path.read_text())
gt = json.loads(gt_path.read_text())

print("Kiem thu vat ly thanh cong!")
print("- QC passed:", qc["report"]["passed"])
print("- So vat the bi va cham:", qc["report"]["metrics"]["factual_struck"])
print("- So vat the bystander khong va cham:", qc["report"]["metrics"]["factual_untouched"])
print("- Hard affected objects:", gt["hard_affected"])

## 7. Sinh Dataset Day Du (Vat ly + Blender GPU Render 3 Nhanh)

In [ ]:
dataset_dir = work_path / "dataset"
gpu_opt = "--require-gpu --strict" if REQUIRE_GPU else ""
build_cmd = (
    f"{runner} scripts/build_velocity_dataset.py --output {dataset_dir} "
    f"--seed {MASTER_SEED} --count {SCENE_COUNT} --resolution {RESOLUTION} "
    f"--samples {SAMPLES_PER_PIXEL} --layers rgba segmentation depth {gpu_opt}"
)

print(f"Bat dau sinh {SCENE_COUNT} scenes (moi scene gom factual, counterfactual, subject_removed)...")
t0 = time.time()
build_env = dict(os.environ, PYTHONPATH=str(repo_dir), TF_CPP_MIN_LOG_LEVEL="3", KUBRIC_USE_GPU="true")
subprocess.run(build_cmd, shell=True, cwd=str(repo_dir), env=build_env, check=True)
print(f"Hoan thanh sinh dataset sau {time.time() - t0:.1f} giay! Thu muc luu tru: {dataset_dir}")

## 8. Kiem tra Output (Video, Mask, Graph, Tracking)

In [ ]:
manifest_file = dataset_dir / "manifest.jsonl"
if manifest_file.exists():
    rows = [json.loads(l) for l in manifest_file.read_text().splitlines() if l.strip()]
    df = pd.DataFrame([{
        "index": r["index"],
        "split": r["split"],
        "objects": r["object_count"],
        "subject": r["subject_shape"],
        "struck": ",".join(r["factual_struck"]),
        "untouched": ",".join(r["factual_untouched"]),
        "hard_affected": ",".join(r["hard_affected"]),
        "renders": ",".join(r["rendered_branches"])
    } for r in rows])
    print("Danh sach cac instances da tao:")
    display(df)
    
    # Kiem tra cac file cua scene dau tien
    first_dir = dataset_dir / rows[0]["path"]
    print(f"\nDanh sach file trong scene 0 ({first_dir}):")
    for p in sorted(first_dir.glob("**/*")):
        if p.is_file():
            print(f"  {p.relative_to(first_dir)} ({p.stat().st_size:,} bytes)")

## 9. Upload Dataset len Hugging Face Hub

In [ ]:
if not HF_REPO_ID or "/" not in HF_REPO_ID:
    print("Vui long nhap HF_REPO_ID tai muc 2 theo dinh dang: username/dataset-name")
elif not HF_TOKEN:
    print("Vui long nhap HF_TOKEN (write access token) tai muc 2 de tien hanh upload")
else:
    publish_args = [
        sys.executable, "scripts/publish_velocity_dataset.py",
        "--output", str(dataset_dir),
        "--repo-id", HF_REPO_ID,
        "--seed", str(MASTER_SEED),
    ] + ([] if HF_PRIVATE else ["--public"])
    
    print(f"Dang upload toan bo dataset len Hugging Face: {HF_REPO_ID}...")
    pub_env = dict(os.environ, HF_TOKEN=HF_TOKEN)
    subprocess.run(publish_args, cwd=str(repo_dir), env=pub_env, check=True)
    print(f"Upload hoan tat! Truy cap dataset tai: https://huggingface.co/datasets/{HF_REPO_ID}")